# Training a Nerual Network Potential

This tutorial introduces how to use `MolPot` to train a nnp

## Preparing and loding data

Before we can start training neural networks with `MolPot`, we need to prepare our data. 

In [1]:
import molpot as mpot

In [2]:
# 1. get QM9 dataset
qm9_ds = mpot.dataset.QM9(
    save_dir="data/qm9",
    total=1000,
    batch_size=100,
    device="cpu",
    split={"train": 0.8, "valid": 0.2},
)
# TODO: super ulgy here, change after torchdata settle down
qm9_ds.calc_nblist(5.0)
train_dp, valid_dp = qm9_ds.get_pipeline()
# 2. wrap with DataLoader

train_dl = mpot.DataLoader(train_dp)
valid_dl = mpot.DataLoader(valid_dp)

In [3]:
data_inspector = mpot.inspector.DataInspector(qm9_ds)
data_inspector.summary()

number of data: 1000

                           dataset: QM9                            
┏━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ label ┃      type       ┃   unit    ┃          comment          ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   A   │ <class 'float'> │    GHz    │   rotational_constant_A   │
│   B   │ <class 'float'> │    GHz    │   rotational_constant_B   │
│   C   │ <class 'float'> │    GHz    │   rotational_constant_C   │
│  mu   │ <class 'float'> │   Debye   │       dipole_moment       │
│ alpha │ <class 'float'> │    a0     │ isotropic_polarizability  │
│ homo  │ <class 'float'> │  hartree  │           homo            │
│ lumo  │ <class 'float'> │  hartree  │           lump            │
│  gap  │ <class 'float'> │  hartree  │            gap            │
│  r2   │ <class 'float'> │    a0     │ electronic_spatial_extent │
│ zpve  │ <class 'float'> │  hartree  │           zpve            │
│  U0   │ <class 'float'> │  hartree  │         energy_U0         │
│   U   │ <class 'float'> │  hartree  │         energy_U          │
│   H   │ <class 'float'> │  hartree  │        enthalpy_H         │
│   G   │ <class 'float'> │  hartree  │        free_energy        │
│  Cv   │ <class 'float'> │ cal/mol/K │       heat_capacity       │
└───────┴─────────────────┴───────────┴───────────────────────────┘

In [4]:
# data_inspector.hist('U0')

In [5]:
for batch in train_dl:
    print(batch.keys())
    break

dict_keys([<Alias: n_atoms>, <Alias: idx>, <Alias: A>, <Alias: B>, <Alias: C>, <Alias: mu>, <Alias: alpha>, <Alias: homo>, <Alias: lumo>, <Alias: gap>, <Alias: r2>, <Alias: zpve>, <Alias: U0>, <Alias: U>, <Alias: H>, <Alias: G>, <Alias: Cv>, <Alias: xyz>, <Alias: Z>, <Alias: cell>, <Alias: pbc>, <Alias: pair_i>, <Alias: pair_j>, <Alias: offsets>, <Alias: d_ij>, <Alias: molid>])


## Setting up the model

In [6]:
cutoff = 5.0
n_rbf = 10
arch = mpot.nnp.PiNet(
    depth=4,
    basis_fn=mpot.nnp.GaussianRBF(n_rbf, cutoff),
    cutoff_fn=mpot.nnp.CosineCutoff(cutoff),
    pp_nodes=[16],
    pi_nodes=[16],
    ii_nodes=[16],
    max_atomtypes=10,
)
energy_readout = mpot.nnp.Atomwise(16, 1, [], from_key="p1", to_key="U0")

model = mpot.PotentialSeq("pinet", arch, energy_readout)

In [7]:
import torch

loss_fn = mpot.train.loss.loss_wrapper(input_="U0", target="U0")(torch.nn.MSELoss())
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, 0.994)

trainer = mpot.PotentialTrainer(
    "pinet-qm9",
    model,
    train_dl,
    loss_fn,
    optimizer,
    scheduler,
    valid_dataloader=valid_dl,
    root_dir="runs",
)
trainer.fix.register(
    trainer.Stage.after_iter,
    mpot.train.fix.TensorBoardFix(
        every_n_steps=100,
        log_dir="runs/pinet-qm9",
        outputs=["epoch_speed", "e_mae", "loss", "valid_loss"],
    ),
)
trainer.fix.register(
    trainer.Stage.after_epoch, mpot.train.fix.EpochSpeed(every_n_epochs=1)
)
trainer.fix.register(
    trainer.Stage.after_epoch, mpot.train.fix.StepSpeed(every_n_steps=1)
)
trainer.fix.register(
    trainer.Stage.after_iter,
    mpot.train.fix.MAE(
        output_key="e_mae", every_n_steps=100, result_key="U0", target_key="U0"
    ),
)

['epoch_speed', 'e_mae', 'loss', 'valid_loss']


In [8]:
%load_ext tensorboard
inputs, outputs = trainer.train(1000)